# Python Specialist — Dataset Generator v5 (Scaled to 5,000)

**Fully deterministic. Zero LLM calls. Same schema as v4, scaled 10x with new domains.**

**What's new vs v4:**
- TARGET raised 500 -> 5,000
- New `dsa` domain (data structures & algorithms) sourced from `TheAlgorithms/Python`
- `ml_ds` domain expanded: added pandas, scipy, keras repos (was sklearn-only)
- ~45 GitHub repos total (was ~20), more files pulled per repo
- Domain-fallback fix: plain-Python files with no notable imports (e.g. most DSA code)
  now correctly inherit their source repo's domain tag instead of defaulting to core_python
- HuggingFace cap raised 50 -> 300 per source

**Output schema is UNCHANGED from v4** — same fields (id, task, instruction,
context_code, error, target_code, tests, domain, difficulty, concepts, libraries,
source_repo, content_hash), so this dataset drops straight into your existing
training notebooks, or gets transformed into skeleton/fill/golden-IO formats later.

Runtime: ~20-30 minutes (GitHub rate limiting dominates at this scale).


In [ ]:
# CELL 0 — INSTALL
!pip install -q datasets requests


In [ ]:
# CELL 1 — IMPORTS + CONFIG
import ast, hashlib, json, os, random, re, time, copy
from collections import Counter, defaultdict
from pathlib import Path
import requests
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)

WORK = Path("/content/dataset_build")
WORK.mkdir(parents=True, exist_ok=True)
RAW_CODE = WORK / "raw_code.jsonl"
FINAL = WORK / "dataset_5000.jsonl"

TARGET = 5000

# Domain quotas — enforced during selection. Same schema as v4, new "dsa" domain added,
# ml_ds quota raised substantially (ML/AI/DS coverage requested), everything scaled ~10x.
DOMAIN_QUOTA = {
    "ml_ds": 700,             # pandas, numpy, sklearn, torch, data pipelines
    "dsa": 500,               # data structures & algorithms (TheAlgorithms/Python)
    "api_design": 600,        # FastAPI, Flask, REST, pagination, middleware
    "database": 500,          # SQLAlchemy, ORM, transactions, migrations
    "software_design": 500,   # SOLID, design patterns
    "async_concurrency": 400,
    "reliability": 400,       # retry, circuit breaker, error handling
    "data_engineering": 400,  # ETL, pipelines, batching
    "testing": 300,
    "security": 300,
    "core_python": 400,       # general stdlib / fallback bucket
}
print(f"Target: {TARGET} | Quota sum: {sum(DOMAIN_QUOTA.values())}")



## Part 1 — Library Mappings + AST Detector + Instruction Builder


In [ ]:
# CELL 2 — LIBRARY-TO-DOMAIN MAPPING (120+ libraries)

LIBRARY_DOMAIN = {
    "sqlalchemy": "database", "sqlmodel": "database", "peewee": "database",
    "tortoise": "database", "databases": "database", "alembic": "database",
    "psycopg2": "database", "psycopg": "database", "asyncpg": "database",
    "pymongo": "database", "motor": "database", "mongoengine": "database",
    "redis": "database", "aioredis": "database", "cassandra": "database",
    "elasticsearch": "database", "prisma": "database", "beanie": "database",
    "odmantic": "database", "pynamodb": "database", "sqlite3": "database",
    "pymysql": "database", "aiomysql": "database", "aiosqlite": "database",
    "fastapi": "api_design", "flask": "api_design", "django": "api_design",
    "starlette": "api_design", "sanic": "api_design", "bottle": "api_design",
    "falcon": "api_design", "tornado": "api_design", "uvicorn": "api_design",
    "gunicorn": "api_design", "pydantic": "api_design", "marshmallow": "api_design",
    "graphene": "api_design", "strawberry": "api_design", "flask_restful": "api_design",
    "djangorestframework": "api_design", "rest_framework": "api_design",
    "asyncio": "async_concurrency", "trio": "async_concurrency",
    "anyio": "async_concurrency", "curio": "async_concurrency",
    "threading": "async_concurrency", "multiprocessing": "async_concurrency",
    "concurrent": "async_concurrency", "gevent": "async_concurrency",
    "celery": "async_concurrency", "dramatiq": "async_concurrency",
    "requests": "networking", "httpx": "networking", "urllib": "networking",
    "urllib3": "networking", "aiohttp": "networking", "grpc": "networking",
    "websockets": "networking", "paramiko": "networking",
    "pytest": "testing", "unittest": "testing", "mock": "testing",
    "hypothesis": "testing", "faker": "testing", "factory_boy": "testing",
    "responses": "testing", "respx": "testing",
    "cryptography": "security", "hashlib": "security", "secrets": "security",
    "bcrypt": "security", "passlib": "security", "itsdangerous": "security",
    "jwt": "security", "authlib": "security", "oauthlib": "security",
    "nacl": "security", "pynacl": "security",
    "numpy": "ml_ds", "pandas": "ml_ds", "scipy": "ml_ds",
    "sklearn": "ml_ds", "torch": "ml_ds", "tensorflow": "ml_ds",
    "keras": "ml_ds", "xgboost": "ml_ds", "lightgbm": "ml_ds",
    "matplotlib": "ml_ds", "seaborn": "ml_ds", "plotly": "ml_ds",
    "polars": "ml_ds", "dask": "ml_ds", "transformers": "ml_ds",
    "langchain": "ml_ds", "statsmodels": "ml_ds", "joblib": "ml_ds",
    "nltk": "ml_ds", "spacy": "ml_ds", "gensim": "ml_ds",
    "csv": "data_engineering", "yaml": "data_engineering", "toml": "data_engineering",
    "lxml": "data_engineering", "kafka": "data_engineering", "pika": "data_engineering",
    "boto3": "data_engineering", "airflow": "data_engineering", "prefect": "data_engineering",
    "dagster": "data_engineering", "great_expectations": "data_engineering",
    "tenacity": "reliability", "backoff": "reliability",
    "circuitbreaker": "reliability", "logging": "reliability",
    "structlog": "reliability", "loguru": "reliability", "sentry_sdk": "reliability",
    "typing": "core_python", "dataclasses": "core_python",
    "functools": "core_python", "itertools": "core_python",
    "collections": "core_python", "abc": "core_python",
    "contextlib": "core_python", "enum": "core_python",
    "pathlib": "core_python", "re": "core_python",
    "json": "core_python", "os": "core_python",
    "click": "core_python", "typer": "core_python", "argparse": "core_python",
    "attrs": "core_python", "attr": "core_python",
}
print(f"Library mappings: {len(LIBRARY_DOMAIN)}")



In [ ]:
# CELL 3 — AST PATTERN DETECTOR (unchanged logic from v4, proven working)

def detect_concepts(tree, source):
    concepts = set()
    libs = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                libs.add(alias.name.split(".")[0])
        elif isinstance(node, ast.ImportFrom):
            if node.module:
                libs.add(node.module.split(".")[0])
                names = {a.name for a in node.names}
                if names & {"ABC", "abstractmethod"}:
                    concepts.update(["abstract_base_class", "SOLID", "dependency_inversion"])
                if "Protocol" in names:
                    concepts.update(["protocol", "SOLID", "interface_segregation"])
                if names & {"dataclass", "field"}:
                    concepts.add("dataclasses")
                if names & {"TypeVar", "Generic"}:
                    concepts.update(["generics", "type_system"])
                if "overload" in names:
                    concepts.update(["overload", "type_system"])
                if "contextmanager" in names:
                    concepts.add("context_manager")
                if names & {"wraps", "lru_cache", "cache", "cached_property"}:
                    concepts.add("decorators")
                if "singledispatch" in names:
                    concepts.update(["single_dispatch", "polymorphism"])
                if "Enum" in names:
                    concepts.add("enums")
                if "BaseModel" in names:
                    concepts.update(["pydantic", "validation", "data_modeling"])
                if names & {"Field", "validator", "field_validator", "model_validator"}:
                    concepts.update(["pydantic", "field_validation"])
                if names & {"APIRouter", "FastAPI"}:
                    concepts.update(["rest_api", "routing", "api_design"])
                if "Depends" in names:
                    concepts.update(["dependency_injection", "SOLID"])
                if "HTTPException" in names:
                    concepts.update(["error_contracts", "api_errors"])
                if "Blueprint" in names:
                    concepts.update(["modular_design", "routing"])
                if names & {"Column", "relationship", "mapped_column", "Mapped"}:
                    concepts.update(["orm", "data_modeling", "relationships"])
                if names & {"Mock", "patch", "MagicMock"}:
                    concepts.update(["testing", "mocking"])
                if names & {"Semaphore", "Lock", "Event", "Condition"}:
                    concepts.update(["concurrency", "synchronization"])
                if "TaskGroup" in names:
                    concepts.update(["async_await", "task_group"])
                if "fixture" in names:
                    concepts.update(["testing", "fixtures"])
                if names & {"DataFrame", "Series"}:
                    concepts.update(["dataframe", "data_manipulation"])
                if names & {"array", "ndarray"}:
                    concepts.add("numerical_computing")
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if isinstance(node, ast.AsyncFunctionDef):
                concepts.add("async_await")
            if node.returns:
                concepts.add("type_hints")
            if any(a.annotation for a in node.args.args):
                concepts.add("type_hints")
            if node.decorator_list:
                concepts.add("decorators")
                for dec in node.decorator_list:
                    ds = ast.unparse(dec).lower()
                    if "abstractmethod" in ds:
                        concepts.update(["abstract_base_class", "SOLID"])
                    if any(f".{m}" in ds for m in ["get","post","put","delete","patch"]):
                        concepts.update(["rest_api", "routing", "endpoint"])
                    if "pytest.fixture" in ds:
                        concepts.update(["testing", "fixtures"])
                    if "retry" in ds or "backoff" in ds:
                        concepts.update(["retry", "reliability"])
                    if "cache" in ds:
                        concepts.add("caching")
            for child in ast.walk(node):
                if isinstance(child, (ast.Yield, ast.YieldFrom)):
                    concepts.add("generators")
            # DSA signal: recursive function (calls itself by name)
            for child in ast.walk(node):
                if isinstance(child, ast.Call) and isinstance(child.func, ast.Name) and child.func.id == node.name:
                    concepts.add("recursion")
        elif isinstance(node, ast.ClassDef):
            concepts.add("oop")
            methods = [n.name for n in node.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]
            ms = set(methods)
            for base in node.bases:
                bs = ast.unparse(base)
                if "ABC" in bs:
                    concepts.update(["abstract_base_class", "SOLID"])
                if "Protocol" in bs:
                    concepts.update(["protocol", "SOLID", "interface_segregation"])
                if "BaseModel" in bs:
                    concepts.update(["pydantic", "data_modeling", "validation"])
                if "Enum" in bs:
                    concepts.add("enums")
                if "Exception" in bs or "Error" in bs:
                    concepts.add("custom_exceptions")
                if "Generic" in bs:
                    concepts.update(["generics", "type_system"])
                if "TestCase" in bs:
                    concepts.update(["testing", "unit_test"])
            if "__enter__" in ms and "__exit__" in ms:
                concepts.add("context_manager")
            if "__iter__" in ms and "__next__" in ms:
                concepts.add("iterator_pattern")
            if "__call__" in ms:
                concepts.add("callable_objects")
            if ms & {"subscribe", "unsubscribe", "notify", "register", "emit"}:
                concepts.add("observer_pattern")
            if ms & {"execute", "undo"}:
                concepts.add("command_pattern")
            if "handle" in ms and "set_next" in ms:
                concepts.add("chain_of_responsibility")
            name_l = node.name.lower()
            if "factory" in name_l or (ms & {"create", "build", "make"} and len(ms) <= 5):
                concepts.add("factory_pattern")
            if "strategy" in name_l:
                concepts.add("strategy_pattern")
            if "builder" in name_l:
                concepts.add("builder_pattern")
            if "adapter" in name_l or "wrapper" in name_l:
                concepts.add("adapter_pattern")
            if "facade" in name_l:
                concepts.add("facade_pattern")
            if "singleton" in name_l:
                concepts.add("singleton_pattern")
            if "repository" in name_l or ms >= {"get", "save"} or ms >= {"find", "add"}:
                concepts.update(["repository_pattern", "SOLID"])
            if "service" in name_l:
                concepts.update(["service_layer", "separation_of_concerns"])
            if "middleware" in name_l:
                concepts.update(["middleware", "api_design"])
            # DSA class signal
            if any(k in name_l for k in ["node", "tree", "graph", "heap", "stack", "queue", "linkedlist"]):
                concepts.add("data_structure")
        elif isinstance(node, ast.Try):
            concepts.add("error_handling")
            for h in node.handlers:
                if h.type:
                    exc = ast.unparse(h.type)
                    if "Timeout" in exc:
                        concepts.add("timeout_handling")
                    if any(e in exc for e in ["ConnectionError", "HTTPError"]):
                        concepts.update(["network_error_handling", "reliability"])
                    if "IntegrityError" in exc:
                        concepts.add("database_error_handling")
            if node.finalbody:
                concepts.add("cleanup_pattern")
        elif isinstance(node, (ast.With, ast.AsyncWith)):
            concepts.add("context_manager")
            if isinstance(node, ast.AsyncWith):
                concepts.add("async_context_manager")
        elif isinstance(node, (ast.ListComp, ast.DictComp, ast.SetComp)):
            concepts.add("comprehensions")
        elif isinstance(node, ast.GeneratorExp):
            concepts.update(["generators", "lazy_evaluation"])

    sl = source.lower()
    text_patterns = {
        "retry": ["retry", "reliability"], "circuit_breaker": ["circuit_breaker", "reliability"],
        "backoff": ["exponential_backoff", "reliability"], "middleware": ["middleware", "api_design"],
        "paginate": ["pagination", "api_design"], "pagination": ["pagination", "api_design"],
        "rate_limit": ["rate_limiting", "api_design"], "unit_of_work": ["unit_of_work"],
        "webhook": ["webhooks"], "serialize": ["serialization"],
        "connection_pool": ["connection_pooling"], "pipeline": ["pipeline_pattern", "data_engineering"],
        "batch_process": ["batching", "data_engineering"],
        "binary search": ["binary_search", "dsa"], "quicksort": ["sorting", "dsa"],
        "mergesort": ["sorting", "dsa"], "merge_sort": ["sorting", "dsa"],
        "depth first": ["dfs", "graph_traversal", "dsa"], "breadth first": ["bfs", "graph_traversal", "dsa"],
        "dynamic programming": ["dynamic_programming", "dsa"], "memoiz": ["memoization", "dsa"],
    }
    for keyword, tags in text_patterns.items():
        if keyword in sl:
            concepts.update(tags)
    if "jwt" in sl and "token" in sl:
        concepts.update(["authentication", "jwt", "security"])
    if "password" in sl and "hash" in sl:
        concepts.update(["password_hashing", "security"])
    if "sanitize" in sl or "escape" in sl:
        concepts.update(["sanitization", "security"])
    if "encrypt" in sl or "decrypt" in sl:
        concepts.update(["encryption", "security"])
    return concepts, libs

print("AST pattern detector ready (with DSA + ML signals added).")



In [ ]:
# CELL 4 — DOMAIN + DIFFICULTY + INSTRUCTION BUILDER

def determine_domain(libs, concepts):
    scores = defaultdict(int)
    for lib in libs:
        key = lib.lower().replace("-", "_")
        if key in LIBRARY_DOMAIN:
            scores[LIBRARY_DOMAIN[key]] += 3
    concept_domain = {
        "rest_api": "api_design", "routing": "api_design", "endpoint": "api_design",
        "middleware": "api_design", "pagination": "api_design", "pydantic": "api_design",
        "orm": "database", "database_model": "database", "repository_pattern": "database",
        "unit_of_work": "database", "relationships": "database", "query_building": "database",
        "async_await": "async_concurrency", "task_group": "async_concurrency",
        "concurrency": "async_concurrency", "synchronization": "async_concurrency",
        "retry": "reliability", "circuit_breaker": "reliability", "timeout_handling": "reliability",
        "exponential_backoff": "reliability", "cleanup_pattern": "reliability",
        "testing": "testing", "fixtures": "testing", "mocking": "testing",
        "password_hashing": "security", "authentication": "security", "encryption": "security",
        "jwt": "security", "sanitization": "security",
        "etl": "data_engineering", "batching": "data_engineering", "pipeline_pattern": "data_engineering",
        "factory_pattern": "software_design", "observer_pattern": "software_design",
        "strategy_pattern": "software_design", "builder_pattern": "software_design",
        "singleton_pattern": "software_design", "adapter_pattern": "software_design",
        "command_pattern": "software_design", "facade_pattern": "software_design",
        "SOLID": "software_design", "dependency_injection": "software_design",
        "service_layer": "software_design", "separation_of_concerns": "software_design",
        "dataframe": "ml_ds", "data_manipulation": "ml_ds", "numerical_computing": "ml_ds",
        "binary_search": "dsa", "sorting": "dsa", "dfs": "dsa", "bfs": "dsa",
        "graph_traversal": "dsa", "dynamic_programming": "dsa", "memoization": "dsa",
        "data_structure": "dsa", "recursion": "dsa",
    }
    for c in concepts:
        if c in concept_domain:
            scores[concept_domain[c]] += 1
    return max(scores, key=scores.get) if scores else "core_python"

def determine_difficulty(tree, source):
    lines = len(source.strip().split("\n"))
    depth = sum(1 for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.If, ast.For, ast.While, ast.Try, ast.With)))
    classes = sum(1 for n in ast.walk(tree) if isinstance(n, ast.ClassDef))
    has_generic = "TypeVar" in source or "Generic" in source
    has_async = "async " in source
    has_decorator = "@" in source
    has_nested = source.count("def ") > 2
    score = min(lines // 10, 3) + min(depth // 3, 3) + min(classes, 2)
    score += 1 if has_generic else 0
    score += 1 if has_async else 0
    score += 1 if has_nested else 0
    score += 1 if has_decorator and has_generic else 0
    if score <= 2: return "beginner"
    if score <= 4: return "intermediate"
    if score <= 7: return "advanced"
    return "expert"

def generate_instruction(tree, source, concepts, libs):
    main_node = None
    for node in ast.iter_child_nodes(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            main_node = node
            break
    if not main_node:
        return "Implement the following Python code"

    doc = ast.get_docstring(main_node)
    if doc:
        first = doc.strip().split("\n")[0].strip().rstrip(".")
        if len(first) > 20 and "TODO" not in first:
            return first

    parts = []
    notable = [l for l in libs if l in LIBRARY_DOMAIN and l not in ("typing", "os", "re", "json", "collections", "functools", "enum", "dataclasses", "abc", "__future__")]

    if isinstance(main_node, ast.ClassDef):
        name = re.sub(r"([A-Z])", r" \1", main_node.name).strip().lower()
        pub = [n.name for n in main_node.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and not n.name.startswith("__")]
        pattern_map = [
            ("repository_pattern", f"Implement a {name} using the repository pattern"),
            ("factory_pattern", f"Implement a {name} using the factory pattern"),
            ("observer_pattern", f"Implement a {name} using the observer pattern"),
            ("strategy_pattern", f"Implement a {name} using the strategy pattern"),
            ("builder_pattern", f"Implement a {name} using the builder pattern"),
            ("singleton_pattern", f"Implement a {name} as a singleton"),
            ("adapter_pattern", f"Implement a {name} adapter"),
            ("command_pattern", f"Implement a {name} using the command pattern"),
            ("chain_of_responsibility", f"Implement a {name} using chain of responsibility"),
            ("iterator_pattern", f"Implement a custom iterator {name}"),
            ("context_manager", f"Implement a {name} as a context manager"),
            ("abstract_base_class", f"Define an abstract {name} interface"),
            ("pydantic", f"Create a {name} Pydantic model with validation"),
            ("database_model", f"Define a {name} database model"),
            ("orm", f"Define a {name} ORM model"),
            ("service_layer", f"Implement a {name} service"),
            ("middleware", f"Implement {name} middleware"),
            ("data_structure", f"Implement a {name} data structure"),
        ]
        matched = False
        for concept, instr in pattern_map:
            if concept in concepts:
                parts.append(instr)
                matched = True
                break
        if not matched:
            parts.append(f"Implement a {name} class")
        if pub:
            parts.append(f"with {', '.join(pub[:4])} methods")
    else:
        name = main_node.name.replace("_", " ")
        is_async = isinstance(main_node, ast.AsyncFunctionDef)
        prefix = "Create an async function" if is_async else "Write a function"
        func_map = [
            ("retry", f"{prefix} that retries operations with exponential backoff"),
            ("circuit_breaker", f"{prefix} that implements a circuit breaker"),
            ("rate_limiting", f"{prefix} with rate limiting"),
            ("pagination", f"{prefix} that handles paginated responses"),
            ("caching", f"{prefix} with caching support"),
            ("middleware", f"{prefix} as middleware"),
            ("etl", f"{prefix} for ETL data processing"),
            ("pipeline_pattern", f"{prefix} for data pipeline processing"),
            ("batching", f"{prefix} that processes items in batches"),
            ("authentication", f"{prefix} that handles authentication"),
            ("jwt", f"{prefix} for JWT token handling"),
            ("password_hashing", f"{prefix} for secure password handling"),
            ("validation", f"{prefix} that validates input data"),
            ("serialization", f"{prefix} for data serialization"),
            ("generators", f"{prefix} as a generator that {name}"),
            ("dataframe", f"{prefix} that transforms and analyzes tabular data"),
            ("binary_search", f"{prefix} implementing binary search"),
            ("sorting", f"{prefix} implementing a sorting algorithm"),
            ("dfs", f"{prefix} implementing depth-first traversal"),
            ("bfs", f"{prefix} implementing breadth-first traversal"),
            ("dynamic_programming", f"{prefix} using dynamic programming"),
            ("recursion", f"{prefix} using recursion that {name}"),
        ]
        matched = False
        for concept, instr in func_map:
            if concept in concepts:
                parts.append(instr)
                matched = True
                break
        if not matched:
            if "endpoint" in concepts or "rest_api" in concepts:
                method = "endpoint"
                for dec in getattr(main_node, "decorator_list", []):
                    ds = ast.unparse(dec).lower()
                    for m in ["get", "post", "put", "delete", "patch"]:
                        if f".{m}" in ds:
                            method = f"{m.upper()} endpoint"
                parts.append(f"{prefix} as a {method}")
            elif "decorators" in concepts and "decorator" not in name:
                parts.append(f"Write a decorator that {name}")
            else:
                parts.append(f"{prefix} that {name}")
        typed_args = [f"{a.arg}: {ast.unparse(a.annotation)}" for a in main_node.args.args if a.arg != "self" and a.annotation]
        if typed_args and len(typed_args) <= 3:
            parts.append(f"accepting {', '.join(typed_args)}")

    if notable:
        parts.append(f"using {', '.join(notable[:3])}")
    extras = []
    if "type_hints" in concepts:
        extras.append("type hints")
    if "error_handling" in concepts and "error" not in " ".join(parts).lower():
        extras.append("proper error handling")
    if "SOLID" in concepts and "SOLID" not in " ".join(parts):
        extras.append("SOLID principles")
    if extras:
        parts.append(f"with {', '.join(extras[:2])}")
    return " ".join(parts)

print("Domain + difficulty + instruction builders ready.")



## Part 2 — Collect Python Code (DSA + ML/AI/DS + Libraries + APIs, ~45 repos)


In [ ]:
# CELL 5 — GITHUB REPOS (expanded: DSA + ML/AI/DS + more libraries, ~45 repos)

GITHUB_REPOS = {
    "dsa": [
        "TheAlgorithms/Python",       # THE canonical DSA repo — sorting, searching, trees,
                                       # graphs, DP, strings, math — huge and pure Python
    ],
    "ml_ds": [
        "scikit-learn/scikit-learn",
        "pandas-dev/pandas",
        "scipy/scipy",
        "keras-team/keras",
    ],
    "software_design": [
        "faif/python-patterns",
        "DeiAsPie/design-patterns-python",
        "jackdbd/design-patterns",
        "spencer-luo/PyDesignPattern",
        "dancergraham/HeadFirstDesignPatterns_python",
        "khadijafatima409/oop-solid-design-patterns",
    ],
    "api_design": [
        "tiangolo/full-stack-fastapi-template",
        "wednesday-solutions/python-fastapi",
        "KenMwaura1/Fast-Api-example",
        "mjhea0/flaskr-tdd",
        "encode/starlette",
        "pallets/flask",
        "marshmallow-code/marshmallow",
        "fastapi-admin/fastapi-admin",
    ],
    "database": [
        "MartinKalema/clean-architecture-ddd-python",
        "Mdslino/fastapi-sqlalchemy-async",
        "sqlalchemy/sqlalchemy",
        "coleifer/peewee",
    ],
    "async_concurrency": [
        "encode/httpx",
        "aio-libs/aiohttp",
        "agronholm/anyio",
    ],
    "reliability": [
        "jd/tenacity",
        "YeonwooSung/fastapi-cb",
        "litl/backoff",
    ],
    "testing": [
        "pytest-dev/pytest",
        "HypothesisWorks/hypothesis",
        "getsentry/responses",
    ],
    "data_engineering": [
        "dagster-io/dagster",
        "PrefectHQ/prefect",
    ],
    "security": [
        "pyca/cryptography",
        "mpdavis/python-jose",
        "jpadilla/pyjwt",
    ],
    "core_python": [
        "psf/requests",
        "pallets/click",
        "python-attrs/attrs",
        "pydantic/pydantic",
    ],
}

all_repos = []
for domain, repos in GITHUB_REPOS.items():
    for r in repos:
        all_repos.append({"repo": r, "domain": domain})
print(f"GitHub repos: {len(all_repos)}")
for domain in GITHUB_REPOS:
    print(f"  {domain}: {len(GITHUB_REPOS[domain])} repos")



In [ ]:
# CELL 6 — FETCH FROM GITHUB (max_files raised — bigger target needs more raw material)

def get_python_files(repo, max_files=60):
    files = []
    for branch in ["main", "master"]:
        url = f"https://api.github.com/repos/{repo}/git/trees/{branch}?recursive=1"
        resp = requests.get(url, headers={"Accept": "application/vnd.github.v3+json"})
        if resp.status_code == 200:
            break
    else:
        print(f"  SKIP {repo}: {resp.status_code}")
        return files
    tree = resp.json().get("tree", [])
    skip = ["test_", "_test.", "tests/", "__pycache__", "setup.py", "conftest",
            "migration", "alembic/versions", "__init__.py", "docs/", "examples/",
            "benchmark", "doc/"]
    py = [f for f in tree if f["path"].endswith(".py") and f["type"] == "blob"
          and not any(x in f["path"].lower() for x in skip)]
    random.shuffle(py)
    for f in py[:max_files]:
        try:
            for b in ["main", "master"]:
                r = requests.get(f"https://raw.githubusercontent.com/{repo}/{b}/{f['path']}")
                if r.status_code == 200:
                    break
            if r.status_code == 200 and 200 < len(r.text) < 15000:
                files.append({"code": r.text, "path": f["path"], "repo": repo})
        except Exception:
            continue
        time.sleep(0.12)
    return files

all_files = []
for item in tqdm(all_repos, desc="Fetching GitHub"):
    # DSA repo is huge and has many short, high-value files — pull more from it
    cap = 400 if item["repo"] == "TheAlgorithms/Python" else 60
    files = get_python_files(item["repo"], max_files=cap)
    for f in files:
        f["source_domain"] = item["domain"]
    all_files.extend(files)
    if files:
        print(f"  {item['repo']}: {len(files)} files")

print(f"\nGitHub files: {len(all_files)}")



In [ ]:
# CELL 7 — FETCH FROM HUGGINGFACE (cap raised to match bigger target)

from datasets import load_dataset

hf_code = []
HF_CAP = 300  # raised from 50 (v4) since TARGET is now 10x bigger

print("Loading MBPP...")
try:
    ds = load_dataset("google-research-datasets/mbpp", "full", split="train")
    count = 0
    for row in ds:
        if count >= HF_CAP:
            break
        c = row.get("code", "")
        if c and ("def " in c or "class " in c):
            hf_code.append({
                "code": c, "repo": "mbpp", "path": f"mbpp/{row.get('task_id','')}",
                "source_domain": "dsa",   # MBPP is mostly small algorithmic problems
                "preset_instruction": row.get("text", ""),
                "preset_tests": "\n".join(row.get("test_list", [])),
            })
            count += 1
    print(f"  MBPP: {count} (capped at {HF_CAP})")
except Exception as e:
    print(f"  MBPP: {e}")

print("Loading Self-Instruct StarCoder...")
try:
    ds = load_dataset("codeparrot/self-instruct-starcoder", split="curated")
    count = 0
    for row in ds:
        if count >= HF_CAP:
            break
        c = row.get("output", "")
        if c and ("def " in c or "class " in c):
            hf_code.append({
                "code": c, "repo": "starcoder-si", "path": "self-instruct",
                "source_domain": "core_python",
                "preset_instruction": row.get("instruction", ""),
            })
            count += 1
    print(f"  StarCoder: {count} (capped at {HF_CAP})")
except Exception as e:
    print(f"  StarCoder: {e}")

print(f"\nHuggingFace: {len(hf_code)} (capped)")
print(f"Combined: {len(all_files) + len(hf_code)}")



## Part 3 — Extract, Analyze, Balanced Selection


In [ ]:
# CELL 8 — EXTRACT + ANALYZE
# Domain assignment priority: (1) strong signal from libraries/concepts via determine_domain,
# (2) fallback to the repo's source_domain hint if auto-detection only found "core_python"
#     (weak/no signal) — this is what correctly tags plain-Python DSA files (e.g. TheAlgorithms,
#     which rarely import anything notable) as "dsa" instead of dumping them all into core_python.

def is_high_quality(code_str):
    try:
        tree = ast.parse(code_str)
        compile(tree, "<f>", "exec")
    except Exception:
        return False
    lines = code_str.strip().split("\n")
    if len(lines) < 5 or len(lines) > 60:
        return False
    has_def = any(isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) for n in ast.walk(tree))
    has_class = any(isinstance(n, ast.ClassDef) for n in ast.walk(tree))
    if not (has_def or has_class):
        return False
    for bad in ["TODO", "FIXME", "raise NotImplementedError", "NotImplemented"]:
        if bad in code_str:
            return False
    if code_str.count("pass") > 3 or code_str.count("...") > 1:
        return False
    return True

def extract_units(code_str):
    try:
        tree = ast.parse(code_str)
    except Exception:
        return []
    imports = [ast.unparse(n) for n in ast.iter_child_nodes(tree) if isinstance(n, (ast.Import, ast.ImportFrom))]
    imp_block = "\n".join(imports) + "\n\n" if imports else ""
    units = []
    for node in ast.iter_child_nodes(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            try:
                src = ast.get_source_segment(code_str, node)
                if src and is_high_quality(imp_block + src):
                    units.append(imp_block + src)
            except Exception:
                continue
    return units

def analyze_unit(code_str, info):
    tree = ast.parse(code_str)
    concepts, libs = detect_concepts(tree, code_str)
    auto_domain = determine_domain(libs, concepts)
    # Fallback: if auto-detection found nothing better than the weak default, trust the
    # repo-level domain hint instead (fixes DSA/plain-python files with no notable imports)
    source_hint = info.get("source_domain")
    domain = auto_domain if auto_domain != "core_python" else (source_hint or auto_domain)
    difficulty = determine_difficulty(tree, code_str)
    instruction = info.get("preset_instruction", "") or generate_instruction(tree, code_str, concepts, libs)
    return {
        "code": code_str, "instruction": instruction,
        "tests": info.get("preset_tests", ""),
        "domain": domain, "difficulty": difficulty,
        "concepts": sorted(list(concepts)), "libraries": sorted(list(libs)),
        "source_repo": info.get("repo", ""), "source_path": info.get("path", ""),
    }

analyzed = []
for f in tqdm(all_files, desc="GitHub"):
    if is_high_quality(f["code"]):
        analyzed.append(analyze_unit(f["code"], f))
    else:
        for u in extract_units(f["code"]):
            analyzed.append(analyze_unit(u, f))
for f in tqdm(hf_code, desc="HuggingFace"):
    if is_high_quality(f["code"]):
        analyzed.append(analyze_unit(f["code"], f))
    else:
        for u in extract_units(f["code"]):
            analyzed.append(analyze_unit(u, f))
print(f"\nTotal analyzed: {len(analyzed)}")
print(f"Domains: {dict(Counter(u['domain'] for u in analyzed).most_common())}")



In [ ]:
# CELL 9 — BALANCED SELECTION (enforces domain quotas)

seen = set()
unique = []
for u in analyzed:
    h = hashlib.sha256(u["code"].strip().encode()).hexdigest()
    if h not in seen:
        seen.add(h)
        u["content_hash"] = h
        unique.append(u)

print(f"After dedup: {len(unique)}")

by_domain = defaultdict(list)
for u in unique:
    by_domain[u["domain"]].append(u)

selected = []
for domain, quota in DOMAIN_QUOTA.items():
    candidates = by_domain.get(domain, [])
    random.shuffle(candidates)
    take = min(len(candidates), quota)
    selected.extend(candidates[:take])
    status = "OK" if take >= quota else f"SHORT (need {quota}, have {len(candidates)})"
    print(f"  {domain}: {take}/{quota} {status}")

remaining = TARGET - len(selected)
if remaining > 0:
    selected_hashes = {s["content_hash"] for s in selected}
    extras = [u for u in unique if u["content_hash"] not in selected_hashes]
    random.shuffle(extras)
    selected.extend(extras[:remaining])
    print(f"  fill: +{min(remaining, len(extras))}")

random.shuffle(selected)
selected = selected[:TARGET]
print(f"\nSelected: {len(selected)}")
print(f"Domain balance:")
for d, c in Counter(u["domain"] for u in selected).most_common():
    print(f"  {d}: {c} ({c/len(selected)*100:.0f}%)")

with RAW_CODE.open("w") as f:
    for u in selected:
        f.write(json.dumps(u, ensure_ascii=False) + "\n")
print(f"Saved: {RAW_CODE}")



## Part 4 — Repair & Refactor Variants (deterministic)


In [ ]:
# CELL 10 — REPAIR GENERATOR (varied instructions + 5 mutations, unchanged from v4)

REPAIR_INSTRUCTIONS = {
    "SyntaxError": [
        "Fix the syntax error in this code",
        "This code has a syntax issue, debug and fix it",
        "Correct the syntax problem preventing this code from running",
    ],
    "IndentationError": [
        "Fix the indentation error in this code",
        "This code has wrong indentation, fix it",
        "Debug and correct the indentation issue",
    ],
    "NameError": [
        "Fix the NameError in this code",
        "A variable reference is broken, find and fix it",
        "This code throws a NameError, debug it",
    ],
    "ModuleNotFoundError": [
        "Fix the missing import in this code",
        "This code is missing a required import, add it",
        "An import statement is missing, fix the code",
    ],
    "TypeError": [
        "Fix the TypeError in this code",
        "This code has a type-related bug, fix it",
        "Debug and fix the method call error",
    ],
}

def _remove_colon(code):
    lines = code.split("\n")
    targets = ["def ", "if ", "for ", "while ", "class ", "elif ", "else", "except", "try", "with "]
    for i in range(len(lines)-1, -1, -1):
        if lines[i].rstrip().endswith(":") and any(k in lines[i] for k in targets):
            lines[i] = lines[i].rstrip().rstrip(":")
            broken = "\n".join(lines)
            try:
                ast.parse(broken)
                continue
            except SyntaxError as e:
                return broken, f"SyntaxError: {e.msg} (line {e.lineno})", "SyntaxError"
    return None, None, None

def _remove_import(code):
    lines = code.split("\n")
    imps = [(i, l) for i, l in enumerate(lines) if l.strip().startswith(("import ", "from "))]
    if not imps:
        return None, None, None
    idx, line = random.choice(imps)
    m = re.search(r"(?:import|from)\s+(\w+)", line)
    mod = m.group(1) if m else "module"
    new = [l for i, l in enumerate(lines) if i != idx]
    return "\n".join(new), f"ModuleNotFoundError: No module named '{mod}'", "ModuleNotFoundError"

def _remove_self(code):
    if "self." not in code:
        return None, None, None
    lines = code.split("\n")
    for i in range(len(lines)-1, -1, -1):
        if "self." in lines[i] and "def " not in lines[i]:
            v = re.search(r"self\.(\w+)", lines[i])
            vn = v.group(1) if v else "attr"
            lines[i] = lines[i].replace("self.", "", 1)
            return "\n".join(lines), f"NameError: name '{vn}' is not defined", "NameError"
    return None, None, None

def _wrong_indent(code):
    lines = code.split("\n")
    for i in range(len(lines)-1, 0, -1):
        if lines[i].startswith("    ") and lines[i].strip():
            lines[i] = lines[i][4:]
            broken = "\n".join(lines)
            try:
                ast.parse(broken)
                continue
            except IndentationError as e:
                return broken, f"IndentationError: {e.msg}", "IndentationError"
    return None, None, None

def _missing_parens(code):
    lines = code.split("\n")
    for i in range(len(lines)-1, -1, -1):
        if "()" in lines[i] and "def " not in lines[i] and "class " not in lines[i]:
            lines[i] = lines[i].replace("()", "", 1)
            broken = "\n".join(lines)
            try:
                ast.parse(broken)
                return broken, "TypeError: 'method' object is not subscriptable", "TypeError"
            except Exception:
                continue
    return None, None, None

_MUTATORS = [_remove_colon, _remove_import, _remove_self, _wrong_indent, _missing_parens]

def create_repair(unit):
    random.shuffle(_MUTATORS)
    for m in _MUTATORS:
        broken, error, err_type = m(unit["code"])
        if broken and error and err_type:
            instr = random.choice(REPAIR_INSTRUCTIONS.get(err_type, ["Fix the error in this code"]))
            return {
                "id": "", "task": "repair",
                "instruction": instr,
                "context_code": broken, "error": error,
                "target_code": unit["code"], "tests": "",
                "domain": unit["domain"], "difficulty": unit["difficulty"],
                "concepts": unit["concepts"] + ["debugging", "error_fixing"],
                "libraries": unit["libraries"],
                "source_repo": unit["source_repo"],
                "content_hash": hashlib.sha256(broken.encode()).hexdigest(),
            }
    return None

print("Repair generator ready (varied instructions).")



In [ ]:
# CELL 11 — REFACTOR GENERATOR (5 uglification types, unchanged from v4)

def _strip_types(code):
    type_names = r"(?:str|int|float|bool|list|dict|set|tuple|None|Optional|List|Dict|Tuple|Set|Any|Union|Callable|Iterable|Iterator|Sequence|Mapping)"
    ugly = re.sub(r":\s*" + type_names + r"(?:\[.*?\])?", "", code)
    ugly = re.sub(r"\s*->\s*" + type_names + r"(?:\[.*?\])?", "", ugly)
    if ugly != code:
        return ugly, "add type hints"
    return code, None

def _strip_docs(code):
    pat1 = re.compile(r'"{3}[\s\S]*?"{3}')
    ugly = pat1.sub("", code)
    pat2 = re.compile(r"'{3}[\s\S]*?'{3}")
    ugly = pat2.sub("", ugly)
    ugly = re.sub(r"\n\s*\n\s*\n", "\n\n", ugly)
    if len(ugly) < len(code) - 15:
        return ugly.strip(), "add docstrings"
    return code, None

def _bad_names(code):
    try:
        tree = ast.parse(code)
    except Exception:
        return code, None
    reps = {}
    c = 0
    for node in ast.walk(tree):
        if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
            n = node.id
            if n not in reps and n != "self" and len(n) > 2 and n.islower() and not n.startswith("_"):
                c += 1
                reps[n] = f"x{c}"
    if len(reps) < 2:
        return code, None
    ugly = code
    for old, new in sorted(reps.items(), key=lambda x: -len(x[0])):
        ugly = re.sub(r"\b" + re.escape(old) + r"\b", new, ugly)
    return ugly, "use descriptive variable names"

def _flatten_comp(code):
    m = re.search(r"(\w+)\s*=\s*\[(.+?)\s+for\s+(\w+)\s+in\s+(.+?)\]", code)
    if not m:
        return code, None
    var, expr, item, it = m.groups()
    rep = f"{var} = []\n    for {item} in {it}:\n        {var}.append({expr})"
    return code[:m.start()] + rep + code[m.end():], "use list comprehension"

def _remove_error_handling(code):
    if "try:" not in code or "except" not in code:
        return code, None
    lines = code.split("\n")
    new_lines = []
    skip_depth = 0
    for line in lines:
        stripped = line.strip()
        if stripped.startswith("try:"):
            skip_depth = 1
            continue
        if skip_depth > 0 and (stripped.startswith("except") or stripped.startswith("finally:")):
            continue
        new_lines.append(line)
    ugly = "\n".join(new_lines)
    try:
        ast.parse(ugly)
        if ugly.strip() != code.strip():
            return ugly, "add error handling"
    except Exception:
        pass
    return code, None

_REFACTORS = [_strip_types, _strip_docs, _bad_names, _flatten_comp, _remove_error_handling]

def create_refactor(unit):
    ugly = unit["code"]
    goals = []
    random.shuffle(_REFACTORS)
    for r in _REFACTORS[:random.randint(1, 3)]:
        u, g = r(ugly)
        if g:
            ugly = u
            goals.append(g)
    if not goals or ugly.strip() == unit["code"].strip():
        return None
    try:
        ast.parse(ugly)
    except Exception:
        return None
    return {
        "id": "", "task": "refactor",
        "instruction": f"Refactor this code: {', '.join(goals)}",
        "context_code": ugly, "error": "", "target_code": unit["code"], "tests": "",
        "domain": unit["domain"], "difficulty": unit["difficulty"],
        "concepts": unit["concepts"] + ["refactoring", "clean_code"],
        "libraries": unit["libraries"],
        "source_repo": unit["source_repo"],
        "content_hash": hashlib.sha256(ugly.encode()).hexdigest(),
    }

print("Refactor generator ready (5 types).")



In [ ]:
# CELL 12 — BUILD FINAL DATASET

units = []
with RAW_CODE.open() as f:
    for line in f:
        if line.strip():
            units.append(json.loads(line))

dataset = []
serial = 0

for unit in tqdm(units, desc="Building dataset"):
    serial += 1
    dataset.append({
        "id": f"gen_{serial:05d}", "task": "generate",
        "instruction": unit["instruction"],
        "context_code": "", "error": "",
        "target_code": unit["code"], "tests": unit.get("tests", ""),
        "domain": unit["domain"], "difficulty": unit["difficulty"],
        "concepts": unit["concepts"], "libraries": unit["libraries"],
        "source_repo": unit["source_repo"],
        "content_hash": unit["content_hash"],
    })
    rep = create_repair(unit)
    if rep:
        serial += 1
        rep["id"] = f"rep_{serial:05d}"
        dataset.append(rep)
    ref = create_refactor(unit)
    if ref:
        serial += 1
        ref["id"] = f"ref_{serial:05d}"
        dataset.append(ref)

seen = set()
final = []
for d in dataset:
    if d["content_hash"] not in seen:
        seen.add(d["content_hash"])
        final.append(d)

with FINAL.open("w") as f:
    for d in final:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

print(f"\nTotal: {len(final)}")
print(f"Tasks: {dict(Counter(d['task'] for d in final))}")
print(f"Domains: {dict(Counter(d['domain'] for d in final).most_common(15))}")
print(f"Difficulty: {dict(Counter(d['difficulty'] for d in final))}")



## Part 5 — Quality Audit + Download


In [ ]:
# CELL 13 — QUALITY AUDIT

import pandas as pd
df = pd.DataFrame(final)
print(f"Total: {len(df)}")
print(f"\n-- Tasks --\n{df['task'].value_counts()}")
print(f"\n-- Domains --\n{df['domain'].value_counts()}")
print(f"\n-- Difficulty --\n{df['difficulty'].value_counts()}")

ok = sum(1 for _, r in df.iterrows() if ast.parse(r["target_code"]))
print(f"\nTarget compiles: {ok}/{len(df)}")

if len(df[df["task"] == "repair"]) > 0:
    broken_ct = 0
    for _, r in df[df["task"] == "repair"].iterrows():
        try:
            ast.parse(r["context_code"])
        except Exception:
            broken_ct += 1
    print(f"Repair context broken: {broken_ct}/{len(df[df['task']=='repair'])}")
    print(f"Repair instruction variety: {df[df['task']=='repair']['instruction'].nunique()} unique")

top_concepts = Counter()
for _, r in df.iterrows():
    for c in r["concepts"]:
        top_concepts[c] += 1
print(f"\n-- Top 25 Concepts --")
for c, n in top_concepts.most_common(25):
    print(f"  {c}: {n}")

top_libs = Counter()
for _, r in df.iterrows():
    for lib in r["libraries"]:
        top_libs[lib] += 1
print(f"\n-- Top 15 Libraries --")
for lib, n in top_libs.most_common(15):
    print(f"  {lib}: {n}")

print(f"\n-- Samples --")
for task in ["generate", "repair", "refactor"]:
    sub = df[df["task"] == task]
    if len(sub) == 0:
        continue
    s = sub.iloc[0]
    print(f"\n[{task.upper()}]")
    print(f"  Domain: {s['domain']} | Difficulty: {s['difficulty']}")
    print(f"  Instruction: {s['instruction'][:120]}")
    print(f"  Concepts: {s['concepts'][:5]}")
    print(f"  Code: {s['target_code'][:80]}...")
    if task == "repair":
        print(f"  Error: {s['error']}")

# Show DSA and ML samples specifically since those are the new additions
for domain in ["dsa", "ml_ds"]:
    sub = df[df["domain"] == domain]
    if len(sub) > 0:
        s = sub.iloc[0]
        print(f"\n[{domain.upper()} SAMPLE]")
        print(f"  Instruction: {s['instruction'][:120]}")
        print(f"  Source: {s['source_repo']}")



In [ ]:
# CELL 14 — DOWNLOAD
from google.colab import files
files.download(str(FINAL))
print(f"Dataset: {FINAL}")
print(f"Total: {len(final)} examples")

